In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('/content/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))

Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64



Task 1 — Multi-Series Line Chart with Highlight
What to build: A line chart showing CO2 emissions over time for all Asian countries in the dataset, with one country highlighted.

Requirements:

All countries shown (for context), but only one highlighted in colour — your choice which
All other lines in grey (#DDDDDD), thinner
Highlighted country labelled directly at the end of its line (not in a legend)
Insight title that names the highlighted country and its story

💡 df[df['Region'] == 'Asia'] to filter; use go.Figure() with a loop for per-country control

In [9]:

# Filter Asia
asia = df[df['Region'] == 'Asia']

# List of Asian countries
countries = asia['Country'].unique()

# Choose highlighted country
highlight = "China"

fig = go.Figure()

# Add grey lines for all Asian countries
for c in countries:
    country_df = asia[asia['Country'] == c]
    fig.add_trace(go.Scatter(
        x=country_df['Year'],
        y=country_df['CO2_Mt'],
        mode='lines',
        line=dict(color='#DDDDDD', width=1),
        showlegend=False,
        hovertemplate=c + "<br>Year: %{x}<br>CO₂: %{y} Mt"
    ))

# Add highlighted country (China)
highlight_df = asia[asia['Country'] == highlight]
fig.add_trace(go.Scatter(
    x=highlight_df['Year'],
    y=highlight_df['CO2_Mt'],
    mode='lines+text',
    line=dict(color='#D62728', width=3),
    text=[None]*(len(highlight_df)-1) + [highlight],
    textposition='middle right',
    textfont=dict(color='#D62728', size=14),
    showlegend=False,
    hovertemplate=highlight + "<br>Year: %{x}<br>CO₂: %{y} Mt"
))

# Layout
fig.update_layout(
    title="China’s Rapid Industrial Rise Drove Asia’s CO₂ Emissions Surge (2000–2022)",
    xaxis_title="Year",
    yaxis_title="CO₂ Emissions (Mt)",
    template="simple_white",
    height=550
)

fig.show()



Task 2 — Slopegraph: Regional Change 2000 vs 2022
What to build: A slopegraph comparing average regional CO2 emissions between 2000 and 2022.

Requirements:

One line per region (not per country — aggregate first)
Colour: regions that increased = one colour; decreased = another
Values labelled at both ends of each line
No y-axis tick labels (the endpoint labels make them redundant)
Insight title stating which regions moved most


💡 df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index() then filter to 2000 and 2022 **bold text**

In [8]:


# Aggregate by Region and Year
regional = (
    df.groupby(['Region', 'Year'])['CO2_Mt']
      .mean()
      .reset_index()
)

# Filter only 2000 and 2022
subset = regional[regional['Year'].isin([2000, 2022])]

# Pivot for easier plotting
pivot = subset.pivot(index='Region', columns='Year', values='CO2_Mt').reset_index()

# Determine color: increase = green, decrease = red
pivot['color'] = pivot.apply(lambda row: '#2ca02c' if row[2022] > row[2000] else '#d62728', axis=1)

fig = go.Figure()

# Add one line per region
for _, row in pivot.iterrows():
    fig.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[row[2000], row[2022]],
        mode='lines+text',
        line=dict(color=row['color'], width=3),
        text=[f"{row[2000]:.1f}", f"{row[2022]:.1f}"],
        textposition=['middle left', 'middle right'],
        textfont=dict(size=12, color=row['color']),
        showlegend=False,
        hovertemplate=(
            f"{row['Region']}<br>"
            "Year %{x}: %{y:.1f} Mt"
        )
    ))

# Add region labels separately (cleaner)
for _, row in pivot.iterrows():
    fig.add_trace(go.Scatter(
        x=[1999.5],  # left label
        y=[row[2000]],
        text=[row['Region']],
        mode='text',
        textposition='middle right',
        showlegend=False,
        textfont=dict(size=13)
    ))

# Layout
fig.update_layout(
    title="Asia Shows the Largest CO₂ Surge, While Europe Declines Sharply (2000–2022)",
    xaxis=dict(title="", showticklabels=False),
    yaxis=dict(title="CO₂ Emissions (Mt)", showticklabels=False),
    template="simple_white",
    height=600
)

fig.show()
